# Time-series CV starter

`input/train.csv`を読み込み、年度ベースのexpanding-window CVとseen/unseen診断を確認するための実験用ノートブックです。モデル学習コードは含めていません。

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

# リポジトリのルートとnotebooks/のどちらから起動してもimportできるようにする。
WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIR.parent if WORKING_DIR.name == 'notebooks' else WORKING_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from validation import make_seen_project_mask, make_time_series_cv

INPUT_DIR = PROJECT_ROOT / 'input'
print('PROJECT_ROOT:', PROJECT_ROOT)
print('INPUT_DIR:', INPUT_DIR)

## Configuration

ファイル名と列名を実データに合わせて変更します。

In [ ]:
TRAIN_PATH = INPUT_DIR / 'train.csv'
YEAR_COL = 'project_start_year'
PROJECT_COL = 'project_name'
TARGET_COL = 'target'
N_VALID_YEARS = 3

## Load data

In [ ]:
train = pd.read_csv(TRAIN_PATH)
print('train shape:', train.shape)
display(train.head())

## Create folds

返される`train_idx`と`valid_idx`はラベルindexです。データ抽出には`.loc`を使います。

In [ ]:
folds, diagnostics = make_time_series_cv(
    df=train,
    year_col=YEAR_COL,
    project_col=PROJECT_COL,
    target_col=TARGET_COL,
    n_valid_years=N_VALID_YEARS,
)

display(diagnostics)

In [ ]:
for fold, (train_idx, valid_idx) in enumerate(folds):
    train_fold = train.loc[train_idx]
    valid_fold = train.loc[valid_idx]
    valid_year = valid_fold[YEAR_COL].iloc[0]
    print(
        f'fold={fold}',
        f'valid_year={valid_year}',
        f'train={len(train_fold)}',
        f'valid={len(valid_fold)}',
    )

## Seen / unseen projects

最新foldについて、validationの各`project_name`がtrainingに過去出現しているか確認します。

In [ ]:
latest_train_idx, latest_valid_idx = folds[-1]
latest_seen = make_seen_project_mask(
    df=train,
    train_idx=latest_train_idx,
    valid_idx=latest_valid_idx,
    project_col=PROJECT_COL,
)

display(latest_seen.value_counts(dropna=False).rename('rows').to_frame())

## OOF container

古い年度や異常年度など、validationに一度も使われない行は`NaN`のまま残します。

In [ ]:
oof = np.full(len(train), np.nan, dtype=float)

for fold, (_, valid_idx) in enumerate(folds):
    valid_positions = train.index.get_indexer(valid_idx)
    assert (valid_positions >= 0).all()

    # モデル作成後に予測値を代入する。
    # valid_prediction = model.predict_proba(X_valid)[:, 1]
    # oof[valid_positions] = valid_prediction

print('OOF container shape:', oof.shape)
print('Initial NaN rows:', np.isnan(oof).sum())